# Embed NCERT candidate passages on Kaggle GPU
Attach the private preparation notebook output. Vectors retain candidate status and are for research until content review is complete. Run this notebook again on the full prepared release.


In [1]:
%pip -q install 'transformers==5.22.2' 'huggingface_hub' 'accelerate'

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json, time
from pathlib import Path
import numpy as np
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

assert torch.cuda.is_available(), 'Enable Kaggle GPU'
sources = list(Path('/kaggle/input').rglob('ncert_prepared/candidate_passages.jsonl'))
assert len(sources) == 1, 'Attach exactly one preparation output'
source = sources[0]
passages = [json.loads(line) for line in source.read_text(encoding='utf-8').splitlines()]
assert passages and len({p['passage_id'] for p in passages}) == len(passages)
assert all(p['review_status'] == 'candidate' and p['text'].strip() for p in passages)
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
model_id = 'Qwen/Qwen3-Embedding-0.6B'
revision = '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3'
model_dir = snapshot_download(model_id, revision=revision)
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModel.from_pretrained(model_dir, dtype=torch.float32, device_map={'': 'cuda'}, attn_implementation='eager', local_files_only=True, trust_remote_code=False, use_safetensors=True).eval()
model.config.use_cache = False


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [3]:
start = time.perf_counter()
vectors = []
batch_size = 4
for offset in range(0, len(passages), batch_size):
    texts = [p['text'] for p in passages[offset:offset + batch_size]]
    batch = tokenizer(texts, padding=True, truncation=False, return_tensors='pt')
    assert batch['input_ids'].shape[1] <= 32768, 'Passage exceeds encoder context'
    with torch.inference_mode():
        batch = batch.to('cuda')
        hidden = model(**batch).last_hidden_state
        mask = batch['attention_mask']
        positions = torch.arange(mask.shape[1], device='cuda').expand_as(mask)
        last = positions.masked_fill(mask == 0, -1).max(dim=1).values
        pooled = hidden[torch.arange(len(texts), device='cuda'), last]
        vectors.append(torch.nn.functional.normalize(pooled.float(), dim=1).cpu().numpy())
matrix = np.concatenate(vectors).astype('float32')
assert matrix.shape == (len(passages), 1024)
assert np.isfinite(matrix).all() and np.allclose(np.linalg.norm(matrix, axis=1), 1, atol=1e-5)
output = Path('/kaggle/working/ncert_embeddings')
output.mkdir(exist_ok=True)
np.savez_compressed(output / 'candidate_vectors.npz', vectors=matrix, passage_ids=np.array([p['passage_id'] for p in passages]))
manifest = {'passages': len(passages), 'dimensions': 1024, 'model_id': model_id, 'model_revision': revision,
            'pooling': 'last token', 'normalization': 'L2', 'document_prefix': '',
            'query_prefix': 'Instruct: Given a question, retrieve relevant passages that answer the question\nQuery: ',
            'source_sha256': source_hash, 'status': 'candidate', 'seconds': time.perf_counter() - start}
manifest['vectors_sha256'] = hashlib.sha256((output / 'candidate_vectors.npz').read_bytes()).hexdigest()
(output / 'embedding_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))


{
  "passages": 7221,
  "dimensions": 1024,
  "model_id": "Qwen/Qwen3-Embedding-0.6B",
  "model_revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
  "pooling": "last token",
  "normalization": "L2",
  "document_prefix": "",
  "query_prefix": "Instruct: Given a question, retrieve relevant passages that answer the question\nQuery: ",
  "source_sha256": "48008ffb9e9faf4f298d60a3be50db847dc568f6c075b167ae05b5eccd511a45",
  "status": "candidate",
  "seconds": 946.8706230739999,
  "vectors_sha256": "982909a484bcab25fed43b6cdac96f315ceca2f4ba9fc80fc53314b39bfc6e22"
}
